# Laboratorio — Fine-Tuning de LLMs con QLoRA

**SI7016 · Procesamiento de Lenguaje Natural Aplicado · Sesión 4 **

Este laboratorio implementa, celda por celda, el pipeline completo descrito en
`Lecture04b-finetuning.pdf`: afinar un LLM abierto —
**Qwen/Qwen3-8B** — con **QLoRA** (LoRA + cuantización de 4 bits) sobre el
dataset **SAMSum** (diálogos → resumen breve), usando `SFTTrainer` de la
librería **TRL**.

**Objetivos:**

1. Cargar un LLM de 8B de parámetros en 4-bit y verificar el ahorro de memoria frente a full fine-tuning.
2. Configurar adaptadores LoRA sobre las proyecciones de atención y feed-forward.
3. Formatear un dataset con `tokenizer.apply_chat_template`.
4. Entrenar con `SFTTrainer` y guardar solo los adaptadores.
5. Evaluar el modelo afinado con la métrica ROUGE.
6. Adaptar el mismo pipeline a otra tarea (extracción JSON) cambiando solo el formateo del prompt.
7. (Bonus) Entender la API de `DPOTrainer` para alignment por preferencias.

**Requisitos:** GPU con al menos 12-16 GB de VRAM (una T4 de Google Colab gratuita es suficiente en 4-bit).


## 0. Instalación de dependencias

Todas las librerías necesarias para QLoRA + TRL. En Google Colab, ejecuta esta
celda una sola vez y reinicia el entorno de ejecución si te lo pide.


In [1]:
!pip install -q -U transformers datasets accelerate
!pip install -q -U bitsandbytes peft trl
!pip install -q -U evaluate rouge-score


In [2]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memoria total (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: Tesla T4
Memoria total (GB): 15.6


## 1. Cargar el modelo base en 4-bit (QLoRA)

Usamos `BitsAndBytesConfig` para cuantizar el modelo base a 4 bits (formato
NF4) en el momento de cargarlo. El modelo cuantizado queda **congelado**: no
se le va a calcular gradiente directamente — solo a los adaptadores LoRA que
agregamos en la Parte 2.

Esta es exactamente la corrección que se usó para resolver el error
`TypeError: ...got an unexpected keyword argument 'load_in_4bit'` visto en el
laboratorio de la Sesión 4 Momento 2: **nunca** pasar `load_in_4bit=True`
directo a `from_pretrained()` — siempre a través de `quantization_config=`.


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-8B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True, # Added to allow CPU offloading
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Modelo cargado:", MODEL_NAME)
print("Memoria GPU ocupada (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Modelo cargado: Qwen/Qwen3-8B
Memoria GPU ocupada (GB): 6.09


**Punto de verificación:** con `Qwen3-8B` en 4-bit, la memoria ocupada
debería rondar los 5-6 GB — muy por debajo de los ~28-32 GB que necesitaría
full fine-tuning del mismo modelo en 16/32-bit (ver diapositiva "QLoRA:
memoria de GPU, antes y después" del Momento 3).


## 2. Configurar los adaptadores LoRA

`prepare_model_for_kbit_training` deja el modelo listo para entrenarse en
precisión mixta sobre pesos cuantizados. Luego `LoraConfig` define las
matrices de bajo rango que se van a insertar — aquí, sobre **todas** las
proyecciones de atención (`q_proj, k_proj, v_proj, o_proj`) y de la red
feed-forward (`gate_proj, up_proj, down_proj`), siguiendo la recomendación
estándar para modelos tipo Llama/Qwen.


In [4]:
from peft import LoraConfig, get_peft_model
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16,                  # rango de las matrices B y A — a mayor r, más capacidad y más costo
    lora_alpha=32,         # factor de escalado de la actualización LoRA
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 43,646,976 || all params: 8,234,382,336 || trainable%: 0.5301


**Pregunta para reflexionar:** ¿qué porcentaje de los parámetros totales
del modelo terminó siendo entrenable? Compáralo con la relación "100×-1000×
menos parámetros entrenables" mencionada en la diapositiva de LoRA del
Momento 3 — ¿tu resultado es consistente con ese rango?


## 3. Dataset SAMSum y formateo con chat template

SAMSum es un dataset de diálogos informales (estilo chat) junto con un
resumen humano de cada uno — un caso clásico de fine-tuning para resumen
generativo.

Para que el modelo aprenda el formato de conversación correcto, cada ejemplo
se convierte a la plantilla de chat de Qwen3 con
`tokenizer.apply_chat_template`. Usamos `enable_thinking=False` (equivalente
a `/no_think`) porque para esta tarea queremos una respuesta directa, sin la
cadena de razonamiento explícita que Qwen3 puede generar en modo "thinking".


In [5]:
from datasets import load_dataset

# Subconjunto pequeño para que el entrenamiento sea rápido en una T4
raw_dataset = load_dataset("knkarthick/samsum", split="train[:2000]")
raw_dataset[0]


{'id': '13818513',
 'dialogue': "Amanda: I baked  cookies. Do you want some?\nJerry: Sure!\nAmanda: I'll bring you tomorrow :-)",
 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.'}

### Alternative Dataset: DialogSum

Since the `samsum` dataset seems to be experiencing persistent issues, you can try using the `dialogsum` dataset as an alternative for dialogue summarization. It provides similar data with `dialogue` and `summary` fields.

In [6]:
def format_example(example):
    messages = [
        {"role": "user", "content": f"Resume el siguiente diálogo en 1-2 frases:\n\n{example['dialogue']}"},
        {"role": "assistant", "content": example["summary"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, enable_thinking=False
    )
    return {"text": text}


dataset = raw_dataset.map(format_example, remove_columns=raw_dataset.column_names)
print(dataset[0]["text"][:600])


<|im_start|>user
Resume el siguiente diálogo en 1-2 frases:

Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)<|im_end|>
<|im_start|>assistant
<think>

</think>

Amanda baked cookies and will bring Jerry some tomorrow.<|im_end|>



## 4. Entrenamiento con SFTTrainer

`SFTTrainer` (de la librería TRL) encapsula el ciclo de entrenamiento
supervisado sobre el campo de texto ya formateado. Como el modelo tiene los
adaptadores LoRA activos, **solo esas matrices** acumulan gradiente durante
`trainer.train()` — el resto de los pesos, cuantizados a 4-bit, permanece
congelado.


In [7]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="qwen3-8b-samsum-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    dataset_text_field="text", # Moved from SFTTrainer to SFTConfig
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
)

trainer.train()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
trainer.save_model("qwen3-8b-samsum-lora")  # guarda SOLO los adaptadores LoRA (unos pocos MB)
tokenizer.save_pretrained("qwen3-8b-samsum-lora")
print("Adaptadores guardados en ./qwen3-8b-samsum-lora")


## 5. Inferencia con el modelo afinado

El modelo ya tiene los adaptadores LoRA cargados en memoria (el objeto
`model` de las celdas anteriores), así que podemos generar directamente sobre
un diálogo nuevo, fuera del conjunto de entrenamiento.


In [ ]:
def resumir(dialogo, max_new_tokens=60):
    messages = [{"role": "user", "content": f"Resume el siguiente diálogo en 1-2 frases:\n\n{dialogo}"}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


dialogo_prueba = (
    "Ana: ¿Vas a venir a la reunión de mañana?\n"
    "Luis: Sí, pero voy a llegar 10 minutos tarde, tengo otra cita antes.\n"
    "Ana: Tranquilo, empezamos con lo del presupuesto, eso lo puedes ver luego.\n"
    "Luis: Perfecto, gracias por avisarme."
)

print(resumir(dialogo_prueba))


## 6. Evaluación cuantitativa con ROUGE

ROUGE compara el resumen generado por el modelo contra el resumen de
referencia (escrito por humanos) del dataset, midiendo el solapamiento de
n-gramas. Es la métrica estándar para tareas de resumen — la misma
mencionada en la diapositiva "Ciclo de vida" del Momento 3.


In [ ]:
import evaluate

rouge = evaluate.load("rouge")

test_dataset = load_dataset("samsum", split="test[:50]")  # muestra pequeña para que sea rápido

predictions, references = [], []
for example in test_dataset:
    pred = resumir(example["dialogue"])
    predictions.append(pred)
    references.append(example["summary"])

scores = rouge.compute(predictions=predictions, references=references)
for k, v in scores.items():
    print(f"{k}: {v:.4f}")


**Punto de verificación:** compara estos puntajes ROUGE con los que
obtendrías corriendo `resumir()` sobre el modelo **sin** los adaptadores LoRA
cargados (es decir, el Qwen3-8B base, tal cual, sin fine-tuning). ¿El
fine-tuning mejoró el ROUGE-L? Esa comparación es, en el fondo, la evidencia
cuantitativa de que el fine-tuning funcionó.


## 7. Variante: extracción estructurada (JSON)

Tal como señala la diapositiva "Ciclo de vida en la práctica" del Momento 3,
cambiar de tarea (de resumen a extracción JSON, o a QA sobre SQuAD) no
requiere tocar el pipeline de entrenamiento — **solo el formateo del
prompt**. Todo lo demás (carga en 4-bit, LoraConfig, SFTTrainer) es idéntico.

Esta celda ilustra únicamente el cambio de formateo (no vuelve a entrenar,
para mantener el laboratorio corto); en un caso real, este `format_example`
reemplazaría al de la Parte 3 y se reentrenaría con `SFTTrainer` de la misma
forma.


In [ ]:
def format_example_json(example):
    """Variante: en vez de un resumen en lenguaje natural, se le pide al
    modelo que devuelva un JSON estructurado con los participantes y el tema.
    Mismo dataset (SAMSum), tarea distinta, mismo pipeline de entrenamiento."""
    instruccion = (
        "Extrae del siguiente diálogo un JSON con las claves "
        '"participantes" (lista) y "tema" (string breve).\n\n'
        f"Diálogo:\n{example['dialogue']}"
    )
    messages = [
        {"role": "user", "content": instruccion},
        # En un dataset real, esta salida se construiría con una etiqueta
        # humana o un modelo más grande usado como "teacher" para bootstrapping
        {"role": "assistant", "content": '{"participantes": [...], "tema": "..."}'},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, enable_thinking=False)
    return {"text": text}


print(format_example_json(raw_dataset[0])["text"][:500])


## 8. Bonus: alignment con DPO

SFT enseña al modelo a imitar una única respuesta ideal. **DPO** (Direct
Preference Optimization) enseña, en cambio, a preferir una respuesta sobre
otra — sin necesitar un reward model separado ni un loop de Reinforcement
Learning, a diferencia de RLHF clásico (ver diapositivas "DPO: Direct
Preference Optimization" del Momento 3).

Esta celda muestra la API de `DPOTrainer` sobre un dataset de preferencias
**sintético y minúsculo**, solo para que quede clara la forma del dataset y
del entrenamiento — no para producir un modelo alineado de verdad (eso
requeriría miles de pares de preferencias reales).


In [ ]:
from datasets import Dataset

# Dataset de preferencias: prompt, respuesta preferida (chosen), respuesta rechazada (rejected)
preferencias = Dataset.from_dict({
    "prompt": [
        "Explica qué es la inteligencia artificial en una frase.",
        "¿Cómo le explicarías RAG a alguien sin experiencia técnica?",
    ],
    "chosen": [
        "La inteligencia artificial son sistemas que aprenden patrones de datos para tomar decisiones o generar contenido.",
        "RAG es como darle a un modelo de lenguaje acceso a una biblioteca de documentos para que consulte antes de responder, en vez de confiar solo en lo que memorizó.",
    ],
    "rejected": [
        "IA.",
        "Es una técnica de machine learning.",
    ],
})

# from trl import DPOTrainer, DPOConfig
#
# dpo_config = DPOConfig(
#     output_dir="qwen3-8b-dpo-demo",
#     per_device_train_batch_size=1,
#     num_train_epochs=1,
#     learning_rate=5e-5,
#     beta=0.1,  # controla qué tanto se aleja el modelo del modelo de referencia
# )
#
# dpo_trainer = DPOTrainer(
#     model=model,               # típicamente, el modelo ya afinado con SFT
#     args=dpo_config,
#     train_dataset=preferencias,
#     processing_class=tokenizer,
# )
# dpo_trainer.train()

preferencias


El bloque de `DPOTrainer` queda comentado a propósito: entrenar DPO
sobre un modelo de 8B, aunque sea con LoRA, ya empieza a ser pesado para una
sola T4 gratuita combinado con lo ya entrenado en este notebook. Si quieres
correrlo, hazlo en una sesión nueva (para liberar memoria) y con un dataset
de preferencias real — por ejemplo, `HuggingFaceH4/ultrafeedback_binarized`.


## Cierre y checklist del laboratorio

Si llegaste hasta aquí, ya recorriste el ciclo de vida completo del
fine-tuning descrito en el Momento 3:

- [x] Cargar un LLM de 8B en 4-bit (QLoRA) y verificar el ahorro de memoria
- [x] Configurar adaptadores LoRA sobre atención y feed-forward
- [x] Formatear un dataset con `apply_chat_template`
- [x] Entrenar con `SFTTrainer` y guardar solo los adaptadores
- [x] Generar texto con el modelo afinado
- [x] Evaluar cuantitativamente con ROUGE
- [x] Adaptar el pipeline a otra tarea cambiando solo el formateo del prompt
- [x] Entender la API de `DPOTrainer` para alignment por preferencias

**Para explorar más:**
- Cambia `r` y `lora_alpha` en `LoraConfig` y observa cómo cambia el número de parámetros entrenables y el ROUGE final.
- Prueba con el dataset SQuAD v2 en vez de SAMSum, adaptando `format_example` a un formato de pregunta-respuesta.
- Compara el resultado de este mismo pipeline sobre un modelo más pequeño (p. ej. `Qwen/Qwen3-4B`) — ¿cambia mucho el ROUGE? ¿Cambia mucho el tiempo de entrenamiento?
